# SafeMaint Vision Colab A100 Server

Runs the existing `ai/vision_service` in a **SigLIP2-base NaFlex vector-only experiment** on an A100. The Qwen3-VL code and configuration are retained but disabled by default, so it can be restored later without reverting this notebook. PDF pages and overlapping regions are embedded before field photos retrieve visually similar pages.

The Colab filesystem is ephemeral. After a runtime restart, existing manuals must be reindexed from the SafeMaint document list.

In [ ]:
# Runtime configuration
SAFE_MAINT_REPO_URL = ""  # optional: https://github.com/your-org/your-repo.git
PROJECT_ROOT = "/content/safemaint"

VISION_QWEN_MODEL = "Qwen/Qwen3-VL-4B-Instruct"
VISION_ENABLE_QWEN = "false"  # vector-only experiment; set true to restore VLM verification
VISION_EMBEDDING_MODEL = "google/siglip2-base-patch16-naflex"
VISION_MAX_NEW_TOKENS = "96"
VISION_QWEN_MAX_PIXELS = "401408"  # 28*28*512; lower latency than the local default
VISION_PDF_MAX_PAGES = "2000"  # large equipment catalogs can exceed 500 pages
VISION_CATALOG_INDEX_DIR = "/content/safemaint_catalog_index"
VISION_MODEL_CACHE_DIR = "/content/hf_cache"
VISION_API_KEY = "change-this-shared-demo-token"

NGROK_AUTH_TOKEN = ""  # required: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Validate secrets and confirm that Colab assigned an A100 before downloading models.
import subprocess
if not NGROK_AUTH_TOKEN:
    raise RuntimeError("Set NGROK_AUTH_TOKEN in the configuration cell.")
if VISION_API_KEY == "change-this-shared-demo-token" or len(VISION_API_KEY) < 16:
    raise RuntimeError("Set VISION_API_KEY to a new random value of at least 16 characters.")
gpu_name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
print("GPU:", gpu_name)
if "A100" not in gpu_name:
    raise RuntimeError("Select Runtime > Change runtime type > A100 GPU and reconnect.")

In [ ]:
# Colab already provides CUDA-enabled PyTorch. PaddleOCR is intentionally omitted:
# the fast path runs SigLIP2 first and Qwen3-VL only for ambiguous candidates.
!pip -q install "fastapi>=0.136,<1.0" "uvicorn[standard]>=0.49,<1.0" "transformers>=4.57,<5.0" "accelerate>=1.10,<2.0" "safetensors>=0.4" "pillow>=11,<13" "pypdf>=6,<7" "pypdfium2>=4.30,<5.0" "numpy>=2,<3" "python-multipart>=0.0.20,<1.0" "qwen-vl-utils>=0.0.14,<1.0" "pyngrok>=7,<8"

In [ ]:
# Clone the repository or upload a project zip containing ai/vision_service/.
from pathlib import Path
import shutil
import subprocess

project_root = Path(PROJECT_ROOT)
if project_root.exists():
    print(f"Project already exists: {project_root}")
elif SAFE_MAINT_REPO_URL:
    subprocess.run(["git", "clone", SAFE_MAINT_REPO_URL, str(project_root)], check=True)
else:
    from google.colab import files
    print("Upload a project zip that contains ai/vision_service/.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No project zip uploaded.")
    archive = Path(next(iter(uploaded.keys()))).resolve()
    unpack_dir = Path("/content/safemaint_vision_upload")
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    shutil.unpack_archive(str(archive), str(unpack_dir))
    candidates = [p for p in unpack_dir.rglob("vision_service") if (p / "main.py").exists()]
    if not candidates:
        raise RuntimeError("Could not find ai/vision_service in uploaded zip.")
    source_root = candidates[0].parents[1]
    shutil.copytree(source_root, project_root)

vision_service_dir = project_root / "ai" / "vision_service"
if not (vision_service_dir / "main.py").exists():
    raise RuntimeError(f"vision_service not found: {vision_service_dir}")
print(f"Using project root: {project_root}")

In [ ]:
# Download model snapshots explicitly so failures happen before server startup.
from huggingface_hub import snapshot_download
if VISION_ENABLE_QWEN.lower() == "true":
    snapshot_download(VISION_QWEN_MODEL, cache_dir=VISION_MODEL_CACHE_DIR)
snapshot_download(VISION_EMBEDDING_MODEL, cache_dir=VISION_MODEL_CACHE_DIR)
print("Model downloads complete.")

In [ ]:
# Start the existing SafeMaint vision API and preload both models into A100 memory.
import os
import subprocess
import time
import urllib.request

os.environ.update({
    "VISION_API_KEY": VISION_API_KEY,
    "VISION_QWEN_MODEL": VISION_QWEN_MODEL,
    "VISION_EMBEDDING_MODEL": VISION_EMBEDDING_MODEL,
    "VISION_MODEL_CACHE_DIR": VISION_MODEL_CACHE_DIR,
    "VISION_CATALOG_INDEX_DIR": VISION_CATALOG_INDEX_DIR,
    "VISION_DEVICE": "cuda",
    "VISION_EMBEDDING_DEVICE": "cuda",
    "VISION_LOAD_IN_4BIT": "false",
    "VISION_ENABLE_QWEN": VISION_ENABLE_QWEN,
    "VISION_ENABLE_PADDLE": "false",
    "VISION_PRELOAD_MODELS": "true",
    "VISION_MAX_NEW_TOKENS": VISION_MAX_NEW_TOKENS,
    "VISION_QWEN_MAX_PIXELS": VISION_QWEN_MAX_PIXELS,
    "VISION_PDF_MAX_PAGES": VISION_PDF_MAX_PAGES,
    "HF_HOME": VISION_MODEL_CACHE_DIR,
})
Path(VISION_CATALOG_INDEX_DIR).mkdir(parents=True, exist_ok=True)
subprocess.run("pkill -f 'uvicorn vision_service.main:app' || true", shell=True)
server_log = open("/content/vision_server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "vision_service.main:app", "--host", "127.0.0.1", "--port", "8020"],
    cwd=str(project_root / "ai"), stdout=server_log, stderr=subprocess.STDOUT,
)
deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        raise RuntimeError(Path("/content/vision_server.log").read_text(errors="replace")[-8000:])
    try:
        with urllib.request.urlopen("http://127.0.0.1:8020/health/live", timeout=5) as response:
            if response.status == 200:
                print(response.read().decode())
                break
    except Exception:
        time.sleep(3)
else:
    raise TimeoutError("Vision model preload did not finish within 15 minutes.")
print("Vision service ready; SigLIP2 is resident on the A100. Qwen3-VL enabled:", VISION_ENABLE_QWEN)

In [ ]:
# Expose the API. Keep this Colab session open while SafeMaint is using it.
if not NGROK_AUTH_TOKEN:
    raise RuntimeError("Set NGROK_AUTH_TOKEN in the first configuration cell.")
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8020, "http").public_url
print("Vision public URL:", public_url)
with urllib.request.urlopen(public_url + "/health/live", timeout=30) as response:
    print("Public health:", response.status, response.read().decode())
print("\nSet this in the local SafeMaint .env and restart backend (do not start the local vision container):")
print(f"VISION_SERVICE_URL={public_url}")
print(f"VISION_API_KEY={VISION_API_KEY}")

## Operating notes

- The local backend continues to enforce login, document permissions, ownership, and site scope before forwarding a PDF or image.
- Do not share the ngrok URL. This notebook is for a controlled demo session, not production.
- Upload/index each catalog after the Colab runtime starts. If the runtime restarts, use **비전 재인덱싱** for existing documents.
- Fast candidate retrieval comes from SigLIP2. Qwen3-VL runs only for ambiguous candidates; OCR is disabled in this A100 fast profile.